# Springboard JSON Exercises
Data Science w/ Python Curriculum
Submitted by: Josh Karpen

## Import Relevant Libraries

In [ ]:
import pandas as pd
import json
from pandas.io.json import json_normalize

## Import Data

In [ ]:
df = pd.read_json('data/world_bank_projects.json')

## Exercise 1 - Find Top 10 Countries w/ Most Projects

In [ ]:
#Check what columns are in the dataset... 
#Project_name and countryname could help answer this question
df.info()

In [ ]:
#First check if project_names are actually unique
proj_counts = df['project_name'].value_counts().to_frame().reset_index().sort_values('project_name', ascending=False)
print(proj_counts.head())
print(" ")
print(proj_counts.sum())

Each Project appears only once so we do not have duplicate records.

In [ ]:
#Group df by countryname and get a count of project_name per country
df_country_proj = df.groupby('countryname')['project_name'].count().to_frame().sort_values('project_name', ascending=False)


The top 10 countries with the most projects are:

In [ ]:
print(df_country_proj.head(10))

Africa is a continent, not a country, so this data might need cleaning.

## Exercise 2 - Find Top 10 Major Project Themes

In [ ]:
#Inspect column mjtheme_namecode
df['mjtheme_namecode'].head()

This column appears to be a nested JSON with Codes and Names embedded within, so we cannot do a groupby and count the values just yet. We will need to normalize this column first using json_normalize. First we reload the dataset using json.load() because the dataframe previously created does not work with json_normalize.

In [ ]:
#Reopening the dataset using json.load to get around 'string indices must be integers' error
with open('data/world_bank_projects.json') as file:
    data = json.load(file)
#pass data through json_normalize
df_themes = json_normalize(data, 'mjtheme_namecode', ['id'])
#Check whether it looks better for analysis now
print(df_themes.head())

Now do a groupby operation to view the count of theme Code and Name and sort in descending order.

In [ ]:
df_themes_grouped_name = df_themes.groupby(['code', 'name'])['id'].count().to_frame().sort_values('id', ascending=False) 
print("Top 10 Themes by Code and Name")
print(df_themes_grouped_name.head(10))

This looks good but checking the output two cells higher, we can see that Code 11 does not have a Name in that output, but in the Top 10 Themes list it does have a name. This indicates that some Codes are missing Names, but only in certain rows of the data. So we are likely under-counting values.

In [ ]:
df_themes_grouped = df_themes.groupby('code')['id'].count().to_frame().sort_values('id', ascending=False)
print("Top 10 Themes by Code Only")
print(df_themes_grouped.head(10))


Removing Name from the groupby lets us see the counts by Code alone. We can see that some counts are higher (Code 11 goes from 223 to 250) but the ranking is the same. However it would be nice to get the correct values and be able to see the name at the same time...

# Exercise 3 - Fill In the Missing Names

In order to create a dataframe with the blank names filled in, first we can create a dataframe with only the unique Code and Name values, without blanks.

In [ ]:
#Create a dataframe with Codes and non-blank Names (only filled-in names kept).
#Remove blank names
df_themes_filled_names = df_themes[df_themes.name != '']
#Drop 'id' column
df_themes_filled_names = df_themes_filled_names.drop('id', axis=1)
#Change the index to Code
df_themes_filled_names.set_index('code', inplace=True)
#Remove duplicate rows, leaving only unique Code and Name values
df_themes_filled_names.drop_duplicates(inplace=True)

print(df_themes_filled_names.sort_values('code'))


Next we can merge df_themes_filled_names with df_themes_grouped (the dataframe created earlier with only the Code and counts per code).


In [ ]:
themes_cleaned = pd.concat([df_themes_filled_names, df_themes_grouped], axis=1, join='inner')
themes_cleaned.sort_values('id', ascending=False)

And now we can see the correct counts by Code and Name, with all names filled in.